# Entrenamiento LoRA paso a paso para Mesias

Inspecciona datos, valida configuracion y deja listo el entrenamiento.

In [1]:
from pathlib import Path
import sys


def find_project_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "src").exists() and (candidate / "data").exists():
            return candidate
    raise RuntimeError(
        "No se encontró la raíz de Mesias. Abre este notebook desde cualquier carpeta dentro del proyecto."
    )


project_root = find_project_root()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.training.entrenamiento import EntrenamientoConfig
from src.training.preparacion_datos import load_jsonl_examples, prepare_sft_dataset

dataset_path = project_root / "data" / "dataset_entrenamiento_docente.jsonl"
dataset_path

WindowsPath('C:/Users/joede/Documents/mesias/data/dataset_entrenamiento_docente.jsonl')

## Carga e inspeccion del dataset

Revisa tamaño, formato y una muestra antes de entrenar.

In [2]:
examples = load_jsonl_examples(dataset_path)

len(examples), examples[0], examples[-1]

(12,
 {'system_prompt': 'Eres Mesias, un asistente didactico, tecnico y prudente.',
  'prompt': 'Explica en una frase que es el producto escalar.',
  'response': 'El producto escalar es una operacion entre dos vectores que produce un numero y mide en parte su alineacion.'},
 {'system_prompt': 'Eres Mesias, un asistente didactico, tecnico y prudente.',
  'prompt': 'Por que conviene separar train, validacion y test?',
  'response': 'Porque cada particion cumple una funcion distinta y evita medir el rendimiento con datos contaminados por el propio ajuste.'})

In [ ]:
prepared = prepare_sft_dataset(examples, max_length=256)

summary = {
    "rows": len(prepared),
    "keys": sorted(prepared[0].keys()),
    "input_tokens_first_row": len(prepared[0]["input_ids"]),
}

summary

## Configuracion segura de entrenamiento en CPU

Empieza con valores prudentes y sube parametros poco a poco.

In [ ]:
config = EntrenamientoConfig(
    epochs=1.0,
    train_batch_size=1,
    gradient_accumulation_steps=8,
    dataloader_num_workers=0,
    cpu_threads=4,
    max_length=1024,
    output_dir=project_root / "data" / "checkpoints" / "mesias-qwen25-lora-safe",
)

config

## Preparación del entrenamiento

Comando de preparación sin entrenamiento real.
El lanzador `tools/train_lora_seguro.py` ya funciona tanto desde la raíz como desde la carpeta `tools`.

In [ ]:
tools_command = ".\\venv\\Scripts\\python.exe .\\tools\\train_lora_seguro.py --dataset .\\data\\dataset_entrenamiento_docente.jsonl --prepare-only"
print(tools_command)

## Ejecucion real opcional

Quita `--prepare-only` solo despues de validar datos y configuracion.

## Checklist

- El dataset carga bien.
- `prepare_sft_dataset` devuelve claves y longitudes coherentes.
- La configuracion sigue siendo prudente para CPU.
- Mesias sigue respondiendo despues de preparar o entrenar adaptadores.